<a href="https://colab.research.google.com/github/zvides12/project_1/blob/Limpieza-predictiva-BD/ZuleimaVides.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

#Como Panda me estaba leyendo la base de datos mal, ya que las columnas se cargaban en una sola variable, con el siguiente código realicé la corrección

with open("/content/Emisiones_Sucias_MEZCLADAS_10000.csv", "r", encoding="latin-1") as f:
    for i in range(5):
        print(f.readline())
#Lectura correcta del CSV y conversión a DataFrame
df = pd.read_csv("/content/Emisiones_Sucias_MEZCLADAS_10000.csv")
print(df.shape)
df.head()
df.info()
#Exploro valorees únicos en las columna Engine Size (L) y Fuel Consumption City(L/100 km), ya que ambas estaban almacenadas como tipo object y debían ser numéricas.
df["Engine Size(L)"].unique()[:10]
df["Fuel Consumption City (L/100 km)"].unique()[:10]
#Reviso si hay texto dentro de las columnas que impediría la conversión directa a tipo numérico
df["Fuel Consumption City (L/100 km)"].str.contains("L/100", na=False).sum()
#Elimino textos inncesarios (la unidad " L/100 km")
df["Fuel Consumption City (L/100 km)"] = (df["Fuel Consumption City (L/100 km)"]
    .str.replace(" L/100 km", "", regex=False))
#Verifiqué que el texto se eliminó
df["Fuel Consumption City (L/100 km)"].unique()[:10]

#Ahora si realizo la conversión a tipo numerico de las columnas (utilizo el parámetro errors='coerce' para convertir cualquier valor no convertible en NaN), ambas columnas las convirtí en columnas tipo float
cols_to_convert = ['Engine Size(L)', 'Fuel Consumption City (L/100 km)']
for col in cols_to_convert:
    df[col] = pd.to_numeric(df[col], errors='coerce')

#validé la correcta conversión mediante la revisión de tipos de datos, conteo de valores nulos y estadísticas descriptivas.
df.info()
df.isnull().sum()
df.describe()
df.isnull().sum().sort_values(ascending=False)
#Antes de decidir cómo manejar los valores faltantes, analicé el comportamiento del tamaño del motor (Engine Size) por marca.
#Calcularé la media de Engine Size agrupando por Make para verificar si existían diferencias significativas entre fabricantes.
#El análisis mostró una alta variabilidad entre marcas, lo que justifica sustituir valores faltantes (NaN) por valores estimados utilizando la media estadística por grupo y no mediante una media global.
df_engine_make = (df.dropna(subset=['Engine Size(L)']) .groupby('Make')['Engine Size(L)'] .mean() .sort_values(ascending=False))
df_engine_make.head(10)
df_engine_make.tail(10)
#Ahora me enfocaré solo en Engine Size(L) y en Make. #Crearé un dataframe auxiliar
df_engine = df.loc[:, ['Make', 'Engine Size(L)']]
df_engine = df_engine.dropna()
df_engine.head()
df_engine.shape
#sustitución de datos faltantes por su media grupal
#Tenía 7902 registros y logré recuperar 1684 registros, sumando un total de 9586 Engine Size(L)
df['Engine Size(L)'].isnull().sum()
media_por_marca = df.groupby('Make')['Engine Size(L)'].mean()
df['Engine Size(L)'] = df['Engine Size(L)'].fillna(df['Make'].map(media_por_marca))
df['Engine Size(L)'].isnull().sum()
#Engine Size depende de Make, entonces Make debe estar limpio
df['Make'].unique()[:20]
df['Make'].value_counts().head(10)
df['Make'].isnull().sum()

df_make = (df.dropna(subset=['Make'])
      .groupby('Make')
      .size()
      .reset_index(name='frecuencia'))
total = len(df)
nulos = df['Make'].isnull().sum()
total = total - nulos

dict_porcentaje = {}

for freq, marca in zip(df_make['frecuencia'], df_make['Make']):
    porcentaje = round(freq / total, 4)
    dict_porcentaje[marca] = porcentaje
rellenar = []

for marca, porcentaje in dict_porcentaje.items():
    cantidad = round(nulos * porcentaje)
    for _ in range(cantidad):
        rellenar.append(marca)

while len(rellenar) < nulos:
    rellenar.append(marca)

rellenar = rellenar[:nulos]

df.loc[df['Make'].isna(), 'Make'] = rellenar
df['Make'].isnull().sum()










Make,Model,Vehicle Class,Engine Size(L),Cylinders,Transmission,Fuel Type,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)

PORSCHE,BOXSTER,TWO-SEATER,2.7,6,M6,Z,11.5,7.9,9.9,29,228

CHEVROLET,SONIC,COMPACT,1.8,4,AS6,X,9.6,6.7,8.3,34,191

MITSUBISHI,Mirage,COMPACT,1.2,3,AV,X,6.6,5.6,6.2,46,143

NISSAN,XTERRA 4WD,SUV - SMALL,4,6,M6,X,16,12.3,14.4,20,331

(10000, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Make                              9586 non-null   object 
 1   Model                             9231 non-null   object 
 2   Vehicle Class                     7933 non-null   object 
 3   Engine Size(L)                    8136 non-null   object 
 4   Cylinders                         8550 non-nu

np.int64(0)